# 长度直接测量与不确定度评定

本 Notebook 用于对多次长度直接测量数据进行 A 类不确定度、B 类不确定度及合成不确定度计算，并严格按照“四舍六入五凑偶”规则对测量结果进行科学修约与末位对齐。

---
### 实验原理与数学公式

#### 1. 算术平均值与样本标准差
对 $n$ 次独立等精度直接测量值 $x_1, x_2, \dots, x_n$：
$$\bar{x} = \frac{1}{n} \sum_{i=1}^n x_i$$
$$s = \sqrt{\frac{1}{n-1} \sum_{i=1}^n (x_i - \bar{x})^2}$$

#### 2. A 类、B 类及合成不确定度
* **A 类不确定度**（由有限次测量随机效应引起）：
$$u_A = \frac{s}{\sqrt{n}}$$
* **B 类不确定度**（由测量仪器极限误差 $\Delta_{\mathrm{inst}}$ 引起，假定均匀分布）：
$$u_B = \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}}$$
* **合成标准不确定度**：
$$u = \sqrt{u_A^2 + u_B^2}$$

#### 3. 修约与结果表示
* 不确定度 $u$ 按照“四舍六入五凑偶”原则保留一位有效数字。
* 测量平均值末位与不确定度的有效位数对齐。

In [ ]:
import math
from decimal import Decimal, ROUND_HALF_EVEN
from python.utils import get_decimal_places, scientific_round

print("工具函数加载完成。")

### 1. 测量数据与仪器参数设置
> **提示**：直接在下方修改测量数据列表 `raw_data`（字符串或数值）与仪器分度值/允差 `res_str` 即可。

In [ ]:
# 测量数据列表 (字符串格式保留输入精度)
raw_data = ["10.25", "10.24", "10.26", "10.25", "10.24", "10.25"]

# 仪器分度值 / 仪器误差限 Δ_inst
res_str = "0.05"

print(f"输入测量数据 (n={len(raw_data)}): {raw_data}")
print(f"仪器误差限 Δ_inst: {res_str}")

### 2. 不确定度计算与科学修约

In [ ]:
def calc_uncertainty(data_strs, res_str="0.05"):
    data = [Decimal(str(s)) for s in data_strs]
    res = Decimal(str(res_str))
    n = len(data)
    
    # 1. 原始精度平均值
    places = get_decimal_places(data_strs)
    sum_val = sum(data)
    mean = sum_val / n
    mean_final = mean.quantize(Decimal('1.' + '0' * places), rounding=ROUND_HALF_EVEN)
    
    # 2. 样本标准差 s
    variance = sum((x - mean) ** 2 for x in data) / (n - 1) if n > 1 else Decimal("0")
    s = Decimal(str(math.sqrt(float(variance))))
    
    # 3. A 类不确定度
    u_A = s / Decimal(str(math.sqrt(n))) if n > 1 else Decimal("0")
    
    # 4. B 类不确定度
    u_B = res / Decimal(str(math.sqrt(3)))
    
    # 5. 合成不确定度
    u = Decimal(str(math.sqrt(float(u_A**2 + u_B**2))))
    
    return mean_final, mean, s, u_A, u_B, u

mean_display, mean_raw, s, u_A, u_B, u = calc_uncertainty(raw_data, res_str)
mean_final, u_final = scientific_round(mean_raw, u)

print("=" * 40)
print("         长 度 测 量 不 确 定 度 分 析         ")
print("=" * 40)
print(f"测量次数 n         : {len(raw_data)}")
print(f"算术平均值         : {mean_raw:.6f}")
print(f"样本标准差 s       : {s:.6f}")
print(f"A 类不确定度 u_A   : {u_A:.6f}")
print(f"B 类不确定度 u_B   : {u_B:.6f}")
print(f"合成不确定度 u     : {u:.6f}")
print("-" * 40)
print(f"★ 最终修约结果    : L = {mean_final} ± {u_final}")
print("=" * 40)